In [17]:
import pandas as pd

df_test = pd.read_parquet("../data/processed/df_test.parquet")
df_test.head()

,title,clause_type,question,is_impossible,answer_text,processed,text
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,Highlight the parts (if any) of this contract ...,False,DISTRIBUTOR AGREEMENT,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,...",EXHIBIT 10.6\n\n ...
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,Highlight the parts (if any) of this contract ...,False,Distributor,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,...",EXHIBIT 10.6\n\n ...
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Agreement Date,Highlight the parts (if any) of this contract ...,False,"7th day of September, 1999.","{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,...",EXHIBIT 10.6\n\n ...
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Effective Date,Highlight the parts (if any) of this contract ...,False,The term of this Agreement shall be ten (10)...,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,...",EXHIBIT 10.6\n\n ...
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Expiration Date,Highlight the parts (if any) of this contract ...,False,The term of this Agreement shall be ten (10)...,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,...",EXHIBIT 10.6\n\n ...


In [18]:
from transformers import AutoTokenizer, DistilBertForQuestionAnswering

tokenizer = AutoTokenizer.from_pretrained('../models')
model = DistilBertForQuestionAnswering.from_pretrained('../models')

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 5644.72it/s]


In [19]:
model.config.model_type

'distilbert'

In [20]:
import torch

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")
model = model.to(device)
model.eval()

Device: mps


DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
     

In [21]:
def predict(question, context):
    encoding = tokenizer(question, context, max_length=512, truncation=True, return_offsets_mapping=True)

    input_ids = torch.tensor(encoding['input_ids']).unsqueeze(0).to(device)
    attention_mask = torch.tensor(encoding['attention_mask']).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)

        start_logits = outputs.start_logits
        end_logits = outputs.end_logits

        start_idx = torch.argmax(start_logits, dim=1).item()
        end_idx = torch.argmax(end_logits, dim=1).item()

        answer_tokens = encoding['input_ids'][start_idx:end_idx+1]
        answer = tokenizer.decode(answer_tokens)
    return answer

In [22]:
row = df_test.iloc[0]
predict(row['question'], row['text'])

'distributor agreement'

In [23]:
def exact_match_score(prediction, ground_truth):
    return int(prediction.strip().lower() == ground_truth.strip().lower())

In [24]:
from collections import Counter

def f1_score(prediction, ground_truth):
    prediction_tokens = prediction.lower().split()
    ground_truth_tokens = ground_truth.lower().split()

    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0

    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)

    f1 = (2 * precision * recall) / (precision + recall)
    return f1




In [25]:
print(exact_match_score('distributor agreement', 'DISTRIBUTOR AGREEMENT'))
print(f1_score('distributor agreement', 'DISTRIBUTOR AGREEMENT'))

1
1.0


In [26]:
em_scores = []
f1_scores = []

for _, row in df_test[df_test['is_impossible'] == False].iterrows():
    prediction = predict(row['question'], row['text'])
    ground_truth = row['answer_text']

    em_scores.append(exact_match_score(prediction, ground_truth))
    f1_scores.append(f1_score(prediction, ground_truth))

print(f"Exact Match: {round(sum(em_scores) / len(em_scores), 4)}")
print(f"F1: {round(sum(f1_scores) / len(f1_scores), 4)}")

Exact Match: 0.1075
F1: 0.1337
